# DBZD Phase 0 Kaggle runner
Edit only the parameter cell below. Use roughly 3-4 runs per session.

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/dbzd.git"
GITHUB_SECRET_NAME = "GITHUB_TOKEN"  # optional for public repositories
ARMS = ["dbzd_full"]
SEEDS = [42]  # review run only; do not add arms/seeds before approval
CONFIG = "configs/default.yaml"
EXPERIMENT_REVISION = "phase0_review_r2"
DATAGEN_SCHEMA_VERSION = 2
REVIEW_ONLY = True
RUNS_INPUT_DIR = ""  # e.g. /kaggle/input/dbzd-phase0-runs/runs
KAGGLE_DATASET_SLUG = "YOUR_USERNAME/dbzd-phase0-runs"
DATASET_TITLE = "DBZD Phase 0 runs"
RUN_PROBES = True
AUTO_FIX_P100 = True


In [ ]:
import json, os, shutil, subprocess
from pathlib import Path

work = Path("/kaggle/working")
repo = work / "dbzd"
if repo.exists():
    shutil.rmtree(repo)

clone_url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret(GITHUB_SECRET_NAME)
except Exception:
    token = None
if token and clone_url.startswith("https://"):
    clone_url = clone_url.replace("https://", f"https://x-access-token:{token}@", 1)
subprocess.run(["git", "clone", "--quiet", clone_url, str(repo)], check=True)
os.chdir(repo)
print("Repository ready at", repo)

In [ ]:
subprocess.run(["python", "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], check=True)
compat_command = ["python", "scripts/kaggle_torch_compat.py"]
if AUTO_FIX_P100:
    compat_command.append("--auto-fix-p100")
subprocess.run(compat_command, check=True)

In [ ]:
runs = repo / "runs"
if RUNS_INPUT_DIR:
    source = Path(RUNS_INPUT_DIR)
    if source.exists():
        shutil.copytree(source, runs, dirs_exist_ok=True)
        print("Restored runs from", source)
runs.mkdir(exist_ok=True)

data_dir = repo / "data" / "phase0"
metadata_path = data_dir / "metadata.json"
data_revision = None
if metadata_path.exists():
    data_revision = json.loads(metadata_path.read_text()).get("generator_schema_version")
if data_revision != DATAGEN_SCHEMA_VERSION:
    if data_dir.exists():
        shutil.rmtree(data_dir)
    print("Generating datagen schema", DATAGEN_SCHEMA_VERSION)
    subprocess.run([
        "python", "-m", "datagen", "--output-dir", str(data_dir),
        "--tokenizer", "HuggingFaceTB/SmolLM-135M",
        "--train-n", "40000", "--val-n", "2000", "--test-n", "2000"
    ], check=True)

In [ ]:
import yaml
if REVIEW_ONLY and (ARMS != ["dbzd_full"] or SEEDS != [42]):
    raise RuntimeError("Review gate active: run only dbzd_full seed 42.")

for arm in ARMS:
    for seed in SEEDS:
        run_dir = runs / f"{arm}_s{seed}"
        resolved = run_dir / "resolved_config.yaml"
        if resolved.exists():
            old_revision = (yaml.safe_load(resolved.read_text()) or {}).get("experiment_revision")
            if old_revision != EXPERIMENT_REVISION:
                print("Removing stale run revision", old_revision, "from", run_dir)
                shutil.rmtree(run_dir)
        if (run_dir / "model_final.pt").exists():
            print("Skipping completed", arm, seed)
            if RUN_PROBES and not (run_dir / "probe_summary.json").exists():
                subprocess.run(["python", "probe.py", "--run-dir", str(run_dir)], check=True)
            continue
        command = [
            "python", "train.py", "--config", CONFIG,
            "--data-dir", str(data_dir), "--run-root", str(runs),
            "--arm", arm, "--seed", str(seed)
        ]
        if (run_dir / "checkpoint_latest.pt").exists():
            command.append("--resume")
        print("Running", arm, seed, "resume=" + str("--resume" in command))
        subprocess.run(command, check=True)
        if RUN_PROBES:
            subprocess.run(["python", "probe.py", "--run-dir", str(run_dir)], check=True)

subprocess.run(["python", "analysis.py", "--runs-dir", str(runs)], check=True)

In [ ]:
import json

export = work / "dbzd-runs-export"
if export.exists():
    shutil.rmtree(export)
export.mkdir()
shutil.copytree(runs, export / "runs")
metadata = {
    "title": DATASET_TITLE,
    "id": KAGGLE_DATASET_SLUG,
    "licenses": [{"name": "CC0-1.0"}]
}
(export / "dataset-metadata.json").write_text(json.dumps(metadata, indent=2))
if "YOUR_USERNAME" not in KAGGLE_DATASET_SLUG:
    subprocess.run([
        "kaggle", "datasets", "version", "-p", str(export),
        "-m", "DBZD Phase 0 checkpoint and metrics update", "--dir-mode", "zip"
    ], check=True)
else:
    print("Set KAGGLE_DATASET_SLUG to publish a Dataset version.")
shutil.make_archive(str(work / "dbzd-runs"), "zip", root_dir=runs)
print("Packaged", work / "dbzd-runs.zip")